# L1, Elastic Net, and L2: converged full-batch comparison

Compare C=**0.1,1,10,100,500,1000,2000** and L1 fractions **0,0.25,0.50,0.75,1**: 35 complete configurations / 175 binary fits, plus the frozen original LBFGS/L2 incumbent. Keep the 150 embeddings/scaler, exact DBSCAN targets/raw weights, normalized fitting-weight total 157 per reason, class weighting, cutoff 0.5, and fallback fixed.

SAGA hit its iteration limit on weakly regularized fits; that incomplete attempt is archived in `saga_probe/` and excluded. The final comparison uses SciPy L-BFGS-B on the same convex weighted logistic objective for all penalties. For nonzero L1 fractions, write w=u-v with u,v>=0, so the L1 term is sum(u+v); the intercept is unpenalized. Pure L2 uses ordinary unconstrained coefficients. At optimum the split formulation equals the usual L1/Elastic Net problem. Numerical derivatives, normalized KKT residuals, and cross-checks against tightly fitted scikit-learn L2 plus strongly regularized SAGA L1/Elastic Net verify the implementation.

Require solver success and KKT residual <=1e-7. Use gtol=1e-9, ftol=1e-15, maxiter=20000; retry more tightly based only on training convergence if needed. Penalty/C are shared across five classifiers within each grid setting. Overall score ties retain the incumbent. No new features, pseudo-label generation, validation-label refit, Kaggle submission, commit, or tag.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from scipy.special import expit
from scipy.optimize import minimize,check_grad
import scipy
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from threadpoolctl import threadpool_limits
from IPython.display import display

REASONS=["Safety","Funding","Efficacy","Recruitment","Other"]
FEATURES=[f"emb_{i}" for i in range(150)]
C_GRID=[0.1,1.0,10.0,100.0,500.0,1000.0,2000.0]
L1_FRACTIONS=[0.0,0.25,0.50,0.75,1.0]
OPTIMIZER_GTOLS=[1e-9,1e-10]
MAX_ITER=20000
KKT_LIMIT=1e-7
cwd=Path.cwd()
DATA=next(p for p in [cwd,cwd/"part1",cwd/"AML-Midterm"/"part1"]
          if (p/"part1_train.csv").is_file() or (p/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR=DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
SOURCE=DATA/"experiments/additional_methods"
OUT=DATA/"experiments/logistic_penalties"
for folder in ["datasets","reason_models","models","validation_predictions","submissions"]:(OUT/folder).mkdir(parents=True,exist_ok=True)
train=pd.read_csv(INPUT_DIR/"part1_train.csv");valid=pd.read_csv(INPUT_DIR/"part1_val.csv")
known=train[REASONS].isin([0,1]).all(axis=1).to_numpy()
original=train[REASONS].to_numpy(dtype=int);truth=valid[REASONS].to_numpy(dtype=int)
assert len(train)==1171 and known.sum()==157 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isin(truth,[0,1]).all()
source_record=json.loads((SOURCE/"submission_manifest.json").read_text())
source_model_path=SOURCE/source_record["model_file"];source_dataset_path=SOURCE/"datasets/dbscan_m5_q25.csv"
source_model=joblib.load(source_model_path)
fixed=pd.read_csv(source_dataset_path,float_precision="round_trip")
assert fixed.ID.tolist()==train.ID.tolist()
Y=fixed[REASONS].to_numpy(dtype=int);W=fixed[[f"weight_{r}" for r in REASONS]].to_numpy()
np.testing.assert_array_equal(Y[known],original[known]);np.testing.assert_array_equal(W[known],np.ones((157,5)))
assert np.array_equal(W>0,Y!=-100)
(OUT/"datasets/fixed_source.csv").write_bytes(source_dataset_path.read_bytes())
scaler=source_model["scaler"];assert scaler.n_samples_seen_==157
XS=scaler.transform(train[FEATURES].to_numpy());VS=scaler.transform(valid[FEATURES].to_numpy())
NORMALIZED_W=W*157/W.sum(axis=0)
weights_frame=train[["ID"]].copy()
for j,r in enumerate(REASONS):weights_frame[f"effective_weight_{r}"]=NORMALIZED_W[:,j]
weights_frame.to_csv(OUT/"datasets/normalized_fitting_weights.csv",index=False)
protected_paths=[INPUT_DIR/"part1_train.csv",INPUT_DIR/"part1_val.csv",source_model_path,source_dataset_path,
                 SOURCE/source_record["submission_file"],SOURCE/"labeling_models/dbscan_m5_q25.joblib",SOURCE/"submission_manifest.json"]
protected_hashes={str(p.relative_to(DATA)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
print("35 penalty/C configurations; all data, scaler, source labels/weights, and prediction rules frozen.")


35 penalty/C configurations; all data, scaler, source labels/weights, and prediction rules frozen.


In [2]:
def make_objective(X,y,weights,C,ratio):
    n=X.shape[1];total=weights.sum()
    def value_gradient(theta):
        coefficient=theta[:n] if ratio==0 else theta[:n]-theta[n:2*n]
        margin=X@coefficient+theta[-1];residual=weights*(expit(margin)-y)
        penalty=0.5*(1-ratio)*np.dot(coefficient,coefficient)
        if ratio:penalty+=ratio*(theta[:n].sum()+theta[n:2*n].sum())
        value=(np.dot(weights,np.logaddexp(0,margin)-y*margin)+penalty/C)/total
        grad_coef=(X.T@residual+(1-ratio)*coefficient/C)/total
        grad_intercept=residual.sum()/total
        if ratio:
            lam=ratio/(C*total);gradient=np.r_[grad_coef+lam,-grad_coef+lam,grad_intercept]
        else:gradient=np.r_[grad_coef,grad_intercept]
        return float(value),gradient
    return value_gradient


def diagnostics(coefficient,intercept,X,y,weights,C,ratio):
    margin=X@coefficient+intercept;total=weights.sum();residual=weights*(expit(margin)-y)
    smooth=(X.T@residual+(1-ratio)*coefficient/C)/total;lam=ratio/(C*total)
    nonzero=np.abs(coefficient)>1e-8
    stationarity=np.maximum(np.abs(smooth)-lam,0)
    stationarity[nonzero]=np.abs(smooth[nonzero]+lam*np.sign(coefficient[nonzero]))
    intercept_gradient=abs(float(residual.sum()/total))
    objective=float((np.dot(weights,np.logaddexp(0,margin)-y*margin)+
                   (0.5*(1-ratio)*np.dot(coefficient,coefficient)+ratio*np.abs(coefficient).sum())/C)/total)
    return {"kkt_residual":max(float(stationarity.max()),intercept_gradient),"objective":objective,
            "intercept_gradient":intercept_gradient,"nonzero_coefficients":int(nonzero.sum()),
            "coefficient_norm":float(np.linalg.norm(coefficient))}


def fit_binary(X,y,weights,C,ratio):
    n=X.shape[1];size=n+1 if ratio==0 else 2*n+1
    initial=np.zeros(size);initial[-1]=np.log(np.dot(weights,y)/np.dot(weights,1-y))
    bounds=None if ratio==0 else [(0,None)]*(2*n)+[(None,None)]
    objective=make_objective(X,y,weights,C,ratio);attempts=[]
    for gtol in OPTIMIZER_GTOLS:
        start=time.monotonic()
        result=minimize(objective,initial,method="L-BFGS-B",jac=True,bounds=bounds,
                        options={"gtol":gtol,"ftol":1e-15,"maxiter":MAX_ITER,"maxls":100,"maxcor":30,"maxfun":100000})
        coefficient=result.x[:n] if ratio==0 else result.x[:n]-result.x[n:2*n]
        intercept=float(result.x[-1]);audit=diagnostics(coefficient,intercept,X,y,weights,C,ratio)
        attempts.append({"gtol":gtol,"seconds":time.monotonic()-start,"iterations":int(result.nit),"success":bool(result.success),
                         "message":str(result.message),"split_objective":float(result.fun),**audit})
        if result.success and audit["kkt_residual"]<=KKT_LIMIT:
            np.testing.assert_allclose(float(result.fun),audit["objective"],rtol=1e-6,atol=1e-10)
            return {"coef":coefficient.copy(),"intercept":intercept,"C":C,"l1_fraction":ratio,
                    "optimizer":"scipy_L-BFGS-B","iterations":int(result.nit),"gtol":gtol,**audit},attempts
        initial=result.x
    raise RuntimeError(f"Unconverged convex fit excluded: C={C}, ratio={ratio}, {attempts[-1]}")


def predict_model(model,raw):
    scaled=model["scaler"].transform(np.asarray(raw));probabilities=[]
    for estimator in model["estimators"]:
        if isinstance(estimator,dict):p=expit(scaled@estimator["coef"]+estimator["intercept"])
        else:p=estimator.predict_proba(scaled)[:,list(estimator.classes_).index(1)]
        probabilities.append(p)
    return np.column_stack(probabilities)


def label_predictions(probabilities):
    assert np.isfinite(probabilities).all() and ((probabilities>=0)&(probabilities<=1)).all()
    predicted=(probabilities>=0.5).astype(int);empty=predicted.sum(axis=1)==0
    rows=np.flatnonzero(empty);predicted[rows,probabilities[rows].argmax(axis=1)]=1
    return predicted,empty

reference_probability=predict_model(source_model,valid[FEATURES].to_numpy())
reference_prediction,_=label_predictions(reference_probability)
reference_sample=f1_score(truth,reference_prediction,average="samples",zero_division=0)
reference_macro=f1_score(truth,reference_prediction,average="macro",zero_division=0)
np.testing.assert_allclose(reference_sample,source_record["validation_sample_f1"],atol=1e-14)
# Check the analytic gradient on an interior split-variable point using finite differences.
j=REASONS.index("Funding");indices=np.flatnonzero(W[:,j]>0)[:40]
objective=make_objective(XS[indices],Y[indices,j],NORMALIZED_W[indices,j],1.0,0.5)
point=np.r_[np.full(300,0.05),0.0]
GRADIENT_ERROR=float(check_grad(lambda t:objective(t)[0],lambda t:objective(t)[1],point))
assert GRADIENT_ERROR<1e-5,GRADIENT_ERROR
print("Analytic gradient check error:",GRADIENT_ERROR)
ROWS=[];FIT_ROWS=[];ATTEMPTS=[];MODEL_CACHE={};PROB_CACHE={};BINARY_CACHE={}


def record_candidate(name,model,ratio,C,stage,fit_records):
    probabilities=predict_model(model,valid[FEATURES].to_numpy());prediction,empty=label_predictions(probabilities)
    coefficient_vectors=[e["coef"] if isinstance(e,dict) else e.coef_[0] for e in model["estimators"]]
    iterations=[e["iterations"] if isinstance(e,dict) else int(e.n_iter_.max()) for e in model["estimators"]]
    row={"experiment":name,"stage":stage,"penalty_type":"L2" if ratio==0 else ("L1" if ratio==1 else "ElasticNet"),
         "l1_fraction":ratio,"C":C,"optimizer":"LBFGS_incumbent" if stage=="incumbent" else "full_batch_L-BFGS-B",
         "sample_f1":f1_score(truth,prediction,average="samples",zero_division=0),
         "macro_f1":f1_score(truth,prediction,average="macro",zero_division=0),
         **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,prediction,average=None,zero_division=0))),
         "fallback_trials":int(empty.sum()),"changed_trials_vs_incumbent":int((prediction!=reference_prediction).any(axis=1).sum()),
         "total_nonzero_coefficients":sum(int((abs(v)>1e-8).sum()) for v in coefficient_vectors),
         "maximum_kkt_residual":max(v["kkt_residual"] for v in fit_records) if fit_records else np.nan,
         "maximum_iterations":max(iterations),"kaggle_score":np.nan}
    row["delta_vs_incumbent"]=row["sample_f1"]-reference_sample
    frame=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):frame[f"probability_{r}"]=probabilities[:,j];frame[f"predicted_{r}"]=prediction[:,j]
    frame["used_fallback"]=empty;frame["trial_f1"]=2*(truth*prediction).sum(axis=1)/(truth.sum(axis=1)+prediction.sum(axis=1))
    frame.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    joblib.dump(model,OUT/"models"/f"{name}.joblib",compress=3)
    ROWS.append(row);MODEL_CACHE[name]=model;PROB_CACHE[name]=probabilities
    pd.DataFrame(ROWS).to_csv(OUT/"comparison.csv",index=False)
    return row

record_candidate("incumbent_lbfgs_l2_C1000",source_model,0.0,1000.0,"incumbent",[])


def run_penalty(ratio):
    penalty="L2" if ratio==0 else ("L1" if ratio==1 else "ElasticNet")
    for C in C_GRID:
        name=f"{penalty}_ratio{ratio:g}_C{C:g}".replace(".","p")
        estimators=[];fit_records=[]
        with threadpool_limits(limits=1):
            for j,reason in enumerate(REASONS):
                included=W[:,j]>0
                estimator,attempts=fit_binary(XS[included],Y[included,j],NORMALIZED_W[included,j],C,ratio)
                for row in attempts:ATTEMPTS.append({"experiment":name,"reason":reason,**row})
                record={"experiment":name,"reason":reason,"penalty_type":penalty,"C":C,"l1_fraction":ratio,
                        "training_rows":int(included.sum()),"fitting_weight_total":float(NORMALIZED_W[included,j].sum()),
                        **{key:estimator[key] for key in ["iterations","gtol","kkt_residual","objective","nonzero_coefficients","coefficient_norm"]}}
                FIT_ROWS.append(record);fit_records.append(record);estimators.append(estimator)
                BINARY_CACHE[(reason,C,ratio)]=estimator
                joblib.dump(estimator,OUT/"reason_models"/f"{name}_{reason}.joblib",compress=3)
        model={**source_model,"kind":"elastic_net_logistic_coefficients","estimators":estimators,"parameter":C,"l1_fraction":ratio,
               "penalty_type":penalty,"optimizer":"scipy_L-BFGS-B","source_dataset":"datasets/fixed_source.csv",
               "normalized_weight_file":"datasets/normalized_fitting_weights.csv","prediction_code":"logistic_penalties.ipynb"}
        row=record_candidate(name,model,ratio,C,"penalty_grid",fit_records)
        pd.DataFrame(FIT_ROWS).to_csv(OUT/"fit_audit.csv",index=False);pd.DataFrame(ATTEMPTS).to_csv(OUT/"optimization_attempts.csv",index=False)
        print(f"{name}: sample F1={row['sample_f1']:.6f}, macro F1={row['macro_f1']:.6f}, nonzero={row['total_nonzero_coefficients']}/750, KKT={row['maximum_kkt_residual']:.2g}",flush=True)


Analytic gradient check error: 1.8028738266524143e-07


In [3]:
run_penalty(0.0)


L2_ratio0_C0p1: sample F1=0.841880, macro F1=0.599954, nonzero=750/750, KKT=3.6e-09


L2_ratio0_C1: sample F1=0.858974, macro F1=0.695743, nonzero=750/750, KKT=2.7e-09


L2_ratio0_C10: sample F1=0.867521, macro F1=0.718678, nonzero=750/750, KKT=9.2e-10


L2_ratio0_C100: sample F1=0.854701, macro F1=0.703577, nonzero=750/750, KKT=9.6e-10


L2_ratio0_C500: sample F1=0.854701, macro F1=0.703577, nonzero=750/750, KKT=9.1e-10


L2_ratio0_C1000: sample F1=0.854701, macro F1=0.703577, nonzero=750/750, KKT=8e-10


L2_ratio0_C2000: sample F1=0.854701, macro F1=0.703577, nonzero=750/750, KKT=9.5e-10


In [4]:
run_penalty(0.25)


ElasticNet_ratio0p25_C0p1: sample F1=0.799145, macro F1=0.540124, nonzero=216/750, KKT=6.9e-09


ElasticNet_ratio0p25_C1: sample F1=0.833333, macro F1=0.649131, nonzero=334/750, KKT=3e-09


ElasticNet_ratio0p25_C10: sample F1=0.844017, macro F1=0.684094, nonzero=419/750, KKT=1.3e-09


ElasticNet_ratio0p25_C100: sample F1=0.856838, macro F1=0.697655, nonzero=466/750, KKT=9.8e-10


ElasticNet_ratio0p25_C500: sample F1=0.856838, macro F1=0.697655, nonzero=490/750, KKT=9.4e-10


ElasticNet_ratio0p25_C1000: sample F1=0.856838, macro F1=0.697655, nonzero=498/750, KKT=1e-09


ElasticNet_ratio0p25_C2000: sample F1=0.858974, macro F1=0.708182, nonzero=511/750, KKT=9.9e-10


In [5]:
run_penalty(0.5)


ElasticNet_ratio0p5_C0p1: sample F1=0.820513, macro F1=0.572313, nonzero=122/750, KKT=6.2e-09


ElasticNet_ratio0p5_C1: sample F1=0.811966, macro F1=0.622476, nonzero=217/750, KKT=4.7e-09


ElasticNet_ratio0p5_C10: sample F1=0.835470, macro F1=0.671113, nonzero=266/750, KKT=1.3e-09


ElasticNet_ratio0p5_C100: sample F1=0.844017, macro F1=0.674737, nonzero=331/750, KKT=9.1e-10


ElasticNet_ratio0p5_C500: sample F1=0.844017, macro F1=0.682264, nonzero=351/750, KKT=9.3e-10


ElasticNet_ratio0p5_C1000: sample F1=0.844017, macro F1=0.682264, nonzero=361/750, KKT=9.8e-10


ElasticNet_ratio0p5_C2000: sample F1=0.844017, macro F1=0.682264, nonzero=374/750, KKT=1e-09


In [6]:
run_penalty(0.75)


ElasticNet_ratio0p75_C0p1: sample F1=0.811966, macro F1=0.615419, nonzero=76/750, KKT=1.3e-08


ElasticNet_ratio0p75_C1: sample F1=0.811966, macro F1=0.624273, nonzero=158/750, KKT=5.8e-09


ElasticNet_ratio0p75_C10: sample F1=0.811966, macro F1=0.620159, nonzero=192/750, KKT=1.7e-09


ElasticNet_ratio0p75_C100: sample F1=0.829060, macro F1=0.648406, nonzero=220/750, KKT=9.4e-10


ElasticNet_ratio0p75_C500: sample F1=0.835470, macro F1=0.662692, nonzero=234/750, KKT=9.9e-10


ElasticNet_ratio0p75_C1000: sample F1=0.835470, macro F1=0.662692, nonzero=243/750, KKT=9.7e-10


ElasticNet_ratio0p75_C2000: sample F1=0.839744, macro F1=0.664486, nonzero=248/750, KKT=9.7e-10


In [7]:
run_penalty(1.0)


L1_ratio1_C0p1: sample F1=0.782051, macro F1=0.502378, nonzero=43/750, KKT=1.1e-08


L1_ratio1_C1: sample F1=0.831197, macro F1=0.634762, nonzero=111/750, KKT=6.7e-09


L1_ratio1_C10: sample F1=0.816239, macro F1=0.625571, nonzero=128/750, KKT=2.8e-09


L1_ratio1_C100: sample F1=0.807692, macro F1=0.624428, nonzero=132/750, KKT=9.9e-10


L1_ratio1_C500: sample F1=0.803419, macro F1=0.608655, nonzero=135/750, KKT=9.7e-10


L1_ratio1_C1000: sample F1=0.803419, macro F1=0.608655, nonzero=136/750, KKT=9.4e-10


L1_ratio1_C2000: sample F1=0.803419, macro F1=0.608655, nonzero=137/750, KKT=9.7e-10


In [8]:
grid=pd.DataFrame(ROWS).query("stage=='penalty_grid'");assert len(grid)==35 and len(FIT_ROWS)==175
# Cross-check equivalent objectives against tightly converged scikit-learn fits.
CROSSCHECKS=[]
with threadpool_limits(limits=1):
    for j,reason in enumerate(REASONS):
        included=W[:,j]>0
        native=clone(source_model["estimators"][j]).set_params(tol=1e-9,max_iter=20000)
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            native.fit(XS[included],Y[included,j],sample_weight=NORMALIZED_W[included,j])
        native_diag=diagnostics(native.coef_[0],float(native.intercept_[0]),XS[included],Y[included,j],NORMALIZED_W[included,j],1000.0,0.0)
        ours=BINARY_CACHE[(reason,1000.0,0.0)]
        assert abs(native_diag["objective"]-ours["objective"])<1e-8
        CROSSCHECKS.append({"reason":reason,"penalty":"L2","C":1000,"native_solver":"lbfgs_tight",
                            "native_objective":native_diag["objective"],"full_batch_objective":ours["objective"],"objective_difference":ours["objective"]-native_diag["objective"]})
    for ratio in [0.5,1.0]:
        reason="Funding";j=REASONS.index(reason);included=W[:,j]>0
        native=LogisticRegression(C=1.0,l1_ratio=ratio,solver="saga",max_iter=100000,tol=1e-7,random_state=42)
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            native.fit(XS[included],Y[included,j],sample_weight=NORMALIZED_W[included,j])
        native_diag=diagnostics(native.coef_[0],float(native.intercept_[0]),XS[included],Y[included,j],NORMALIZED_W[included,j],1.0,ratio)
        ours=BINARY_CACHE[(reason,1.0,ratio)]
        assert abs(native_diag["objective"]-ours["objective"])<1e-8
        CROSSCHECKS.append({"reason":reason,"penalty":"L1" if ratio==1 else "ElasticNet","C":1,"native_solver":"saga_strong_reg",
                            "native_objective":native_diag["objective"],"full_batch_objective":ours["objective"],"objective_difference":ours["objective"]-native_diag["objective"]})
pd.DataFrame(CROSSCHECKS).to_csv(OUT/"solver_crosschecks.csv",index=False)
comparison=pd.DataFrame(ROWS)
matched=grid.query("l1_fraction==0").set_index("C").sample_f1.to_dict()
comparison["delta_vs_same_C_converged_L2"]=[row.sample_f1-matched[row.C] for row in comparison.itertuples(index=False)]
comparison.to_csv(OUT/"comparison.csv",index=False)
ranked=comparison.assign(is_incumbent=comparison.stage.eq("incumbent"),distance_C=abs(np.log10(comparison.C/1000))).sort_values(
    ["sample_f1","is_incumbent","l1_fraction","distance_C","C"],ascending=[False,False,True,True,True],kind="stable")
best=ranked.iloc[0];best_by_penalty=ranked.groupby("penalty_type",sort=False).head(1)
best_by_penalty.to_csv(OUT/"best_by_penalty.csv",index=False)
selected=joblib.load(OUT/"models"/f"{best.experiment}.joblib");selected_probability=predict_model(selected,valid[FEATURES].to_numpy())
for row in comparison.itertuples(index=False):
    model=joblib.load(OUT/"models"/f"{row.experiment}.joblib")
    frame=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==valid.ID.tolist()
    probs=frame[[f"probability_{r}" for r in REASONS]].to_numpy();prediction=frame[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    np.testing.assert_allclose(predict_model(model,valid[FEATURES].to_numpy()),probs,rtol=1e-12,atol=1e-12)
    np.testing.assert_array_equal(label_predictions(probs)[0],prediction)
    np.testing.assert_allclose(f1_score(truth,prediction,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,prediction,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
frame=pd.read_csv(OUT/"datasets/fixed_source.csv",float_precision="round_trip")
weights=pd.read_csv(OUT/"datasets/normalized_fitting_weights.csv",float_precision="round_trip")
assert frame.ID.tolist()==train.ID.tolist()==weights.ID.tolist()
np.testing.assert_array_equal(frame[REASONS].to_numpy(dtype=int),Y)
np.testing.assert_array_equal(frame[[f"weight_{r}" for r in REASONS]].to_numpy(),W)
scaled=scaler.transform(frame[FEATURES].to_numpy());replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j,reason in enumerate(REASONS):
        w=weights[f"effective_weight_{reason}"].to_numpy();included=w>0
        np.testing.assert_allclose(w,NORMALIZED_W[:,j],rtol=1e-12,atol=1e-12)
        if isinstance(selected["estimators"][j],dict):
            estimator,_=fit_binary(scaled[included],Y[included,j],w[included],float(best.C),float(best.l1_fraction))
        else:
            estimator=clone(selected["estimators"][j])
            with warnings.catch_warnings():
                warnings.simplefilter("error",ConvergenceWarning)
                estimator.fit(scaled[included],Y[included,j],sample_weight=w[included])
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_model(replay,valid[FEATURES].to_numpy()),selected_probability,rtol=1e-5,atol=1e-7)
np.testing.assert_array_equal(label_predictions(predict_model(replay,valid[FEATURES].to_numpy()))[0],label_predictions(selected_probability)[0])
for rel,h in protected_hashes.items():assert hashlib.sha256((DATA/rel).read_bytes()).hexdigest()==h
print("All 175 convex fits converged; finite-difference, native-solver cross-checks, saved models/scores and selected refit verified.")
print("Best:",best.experiment,"sample F1:",best.sample_f1,"macro F1:",best.macro_f1)
display(best_by_penalty[["penalty_type","l1_fraction","C","sample_f1","macro_f1","delta_vs_incumbent","delta_vs_same_C_converged_L2"]].round(6))


All 175 convex fits converged; finite-difference, native-solver cross-checks, saved models/scores and selected refit verified.
Best: incumbent_lbfgs_l2_C1000 sample F1: 0.9102564102564102 macro F1: 0.811421460284173


,penalty_type,l1_fraction,C,sample_f1,macro_f1,delta_vs_incumbent,delta_vs_same_C_converged_L2
0,L2,0.00,1000.0,0.910256,0.811421,0.000000,0.055556
14,ElasticNet,0.25,2000.0,0.858974,0.708182,-0.051282,0.004274
30,L1,1.00,1.0,0.831197,0.634762,-0.079060,-0.027778


In [9]:
candidate=None
if best.sample_f1>reference_sample+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    prediction,empty=label_predictions(predict_model(selected,test[FEATURES].to_numpy()))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_dbscan_{best.experiment}_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"penalty_type":best.penalty_type,"l1_fraction":float(best.l1_fraction),"C":float(best.C),
               "validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"C_grid":C_GRID,"l1_fraction_grid":L1_FRACTIONS,"grid_configurations":35,"binary_grid_fits":175,
          "optimizer":"SciPy L-BFGS-B full batch; split nonnegative coefficients for nonzero L1",
          "optimizer_gtols":OPTIMIZER_GTOLS,"ftol":1e-15,"max_iter":MAX_ITER,"kkt_acceptance_limit":KKT_LIMIT,
          "gradient_finite_difference_error":GRADIENT_ERROR,"native_solver_crosschecks":len(CROSSCHECKS),
          "saga_probe":"saga_probe/saga_probe.ipynb; incomplete iteration-limited attempt, excluded from ranking",
          "total_complete_configurations":len(comparison),"features":FEATURES,"classifier_scaler_fit_rows":157,
          "source_targets_changed":False,"source_raw_weights_changed":False,"fitting_weight_total_per_reason":157,
          "class_weight":None,"cutoff":0.5,"fallback":True,"validation_rows":78,"validation_used_for_fit":False,
          "final_validation_label_refit":False,"reference_sample_f1":reference_sample,"reference_macro_f1":reference_macro,
          "best_experiment":best.experiment,"best_sample_f1":float(best.sample_f1),"best_macro_f1":float(best.macro_f1),
          "best_penalty_type":best.penalty_type,"best_C":float(best.C),"best_l1_fraction":float(best.l1_fraction),
          "leader_changed":bool(best.sample_f1>reference_sample+1e-12),"candidate":candidate,"protected_input_sha256":protected_hashes,
          "dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*.csv'))},
          "selection":"sample F1; ties retain incumbent, then smaller L1 fraction and C closest to 1000",
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"scipy":scipy.__version__,"numpy":np.__version__,"pandas":pd.__version__},"new_kaggle_scores":None}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
def table(rows):
    lines=["| Penalty | L1 fraction | C | Optimizer | Sample F1 | Macro F1 | Nonzero /750 | Change vs incumbent | Change vs matched L2 |",
           "|---|---:|---:|---|---:|---:|---:|---:|---:|"]
    for row in rows.itertuples(index=False):
        lines.append(f"| {row.penalty_type} | {row.l1_fraction:g} | {row.C:g} | {row.optimizer} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.total_nonzero_coefficients} | {row.delta_vs_incumbent:+.6f} | {row.delta_vs_same_C_converged_L2:+.6f} |")
    return "\n".join(lines)
lines=["# Logistic penalty-type comparison","",
       "35 settings cross C=0.1/1/10/100/500/1000/2000 with L1 fractions 0/0.25/0.50/0.75/1. "
       "Zero is L2, one is L1, intermediate values are Elastic Net. The saved original LBFGS/L2 model remains incumbent.","",
       "## Numerical optimization","",
       "An initial SAGA attempt hit max_iter=100000 on a weakly regularized fit and is archived under saga_probe/, "
       "excluded from ranking. Final fits solve the same weighted logistic objective with full-batch L-BFGS-B. "
       "For nonzero L1 fraction, write w=u-v with nonnegative u,v and penalize sum(u+v); optimal splitting yields the "
       "ordinary L1 norm. The intercept is unpenalized. Pure L2 uses unconstrained coefficients. Loss and penalty are "
       "normalized consistently with scikit-learn: weighted loss sum plus penalty/C, divided by total fitting weight.","",
       f"All 175 fits require optimizer success and normalized KKT residual <={KKT_LIMIT:g}. "
       f"Finite-difference gradient check error: {GRADIENT_ERROR:.3g}. Seven native-solver objective checks compare "
       "against tight scikit-learn LBFGS/L2 for all reasons and converged SAGA L1/Elastic Net at C=1 for Funding. "
       "Matched full-batch L2 controls distinguish penalty changes from solver/convergence changes. Fitted coefficient "
       "dictionaries are portable; predict_model in logistic_penalties.ipynb applies the usual sigmoid decision function.","",
       f"Incumbent sample F1 **{reference_sample:.6f}**, macro F1 **{reference_macro:.6f}**. "
       f"Selected **{best.experiment}**: sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "## Best by penalty family","",table(best_by_penalty),"",
       "## All configurations","",table(ranked),"",
       "## Controls, artifacts, and interpretation","",
       "Keep the exact 150-dimensional embedding/scaler, DBSCAN targets/raw weights, normalized fitting-weight total "
       "157 per reason, class_weight=None, cutoff 0.5, and highest-probability fallback fixed. Penalty and C are shared "
       "across the five reason classifiers within a configuration. No validation/test labels enter fitting and no "
       "final train+validation refit is performed.","",
       "Every saved model reloads with matching probabilities and rescored sample/macro F1. The selected model "
       "independently refits from persisted data/normalized weights with matching predictions. Source targets, raw "
       "weights, and protected artifacts retain their checksums. Randomly rotated embeddings do not inherently favor "
       "sparse coordinate selection. Scores are exploratory selections on the repeatedly consulted 78 validation trials, "
       "not independent test estimates or an exhaustive per-reason penalty/C search.","",
       "datasets/ saves shared fixed data and normalized weights; reason_models/ saves all 175 coefficient fits; "
       "models/ and validation_predictions/ preserve full configurations. fit_audit.csv, optimization_attempts.csv, "
       "solver_crosschecks.csv, comparison.csv, best_by_penalty.csv, and settings.json preserve convergence, sparsity, "
       "scores, and checksums. saga_probe/ retains the incomplete first numerical attempt for transparency.","",
       "No Kaggle submission, commit, or tag was made.",""]
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No setting improves primary sample F1; retain the existing L2 model and its prepared Kaggle file.")
(OUT/"SUMMARY.md").write_text("\n".join(lines)+"\n")
print(table(best_by_penalty))
print("Leader changed:",settings['leader_changed'],"Candidate:",candidate)


| Penalty | L1 fraction | C | Optimizer | Sample F1 | Macro F1 | Nonzero /750 | Change vs incumbent | Change vs matched L2 |
|---|---:|---:|---|---:|---:|---:|---:|---:|
| L2 | 0 | 1000 | LBFGS_incumbent | 0.910256 | 0.811421 | 750 | +0.000000 | +0.055556 |
| ElasticNet | 0.25 | 2000 | full_batch_L-BFGS-B | 0.858974 | 0.708182 | 511 | -0.051282 | +0.004274 |
| L1 | 1 | 1 | full_batch_L-BFGS-B | 0.831197 | 0.634762 | 111 | -0.079060 | -0.027778 |
Leader changed: False Candidate: None


## Native same-solver tolerance diagnostic

Change only the native LBFGS stopping tolerance (the higher iteration limit is not reached). This isolates the numerical-convergence effect, cross-checks the matched full-batch L2 result, and does not select another penalty candidate.

In [10]:
def verify_native_l2_tolerance():
    from pathlib import Path
    import json,warnings
    import numpy as np,pandas as pd,joblib
    from sklearn.base import clone
    from sklearn.metrics import f1_score
    from sklearn.exceptions import ConvergenceWarning
    from threadpoolctl import threadpool_limits
    cwd=Path.cwd()
    root=next(p for p in [cwd,cwd/'part1',cwd/'AML-Midterm/part1'] if (p/'data/part1_train.csv').is_file() or (p/'part1_train.csv').is_file()).resolve()
    raw=root if (root/'part1_train.csv').is_file() else root/'data'
    out=root/'experiments/logistic_penalties';destination=out/'numerical_controls';destination.mkdir(exist_ok=True)
    source=joblib.load(root/'experiments/additional_methods/classifier_models/dbscan_m5_q25__logistic_C1000.joblib')
    train=pd.read_csv(raw/'part1_train.csv');valid=pd.read_csv(raw/'part1_val.csv');fixed=pd.read_csv(out/'datasets/fixed_source.csv',float_precision='round_trip')
    reasons=source['reasons'];features=source['features'];X=source['scaler'].transform(train[features].to_numpy());V=source['scaler'].transform(valid[features].to_numpy())
    model={**source,'estimators':[]};audit=[]
    with threadpool_limits(limits=1):
        for j,reason in enumerate(reasons):
            y=fixed[reason].to_numpy(dtype=int);w=fixed[f'weight_{reason}'].to_numpy();included=w>0;effective=w[included]*157/w[included].sum()
            estimator=clone(source['estimators'][j]).set_params(tol=1e-9,max_iter=20000)
            with warnings.catch_warnings():
                warnings.simplefilter('error',ConvergenceWarning);estimator.fit(X[included],y[included],sample_weight=effective)
            model['estimators'].append(estimator)
            audit.append({'reason':reason,'original_iterations':int(source['estimators'][j].n_iter_.max()),'tight_iterations':int(estimator.n_iter_.max())})
    probabilities=np.column_stack([e.predict_proba(V)[:,list(e.classes_).index(1)] for e in model['estimators']])
    prediction=(probabilities>=.5).astype(int);empty=prediction.sum(axis=1)==0;rows=np.flatnonzero(empty)
    prediction[rows,probabilities[rows].argmax(axis=1)]=1
    truth=valid[reasons].to_numpy(dtype=int)
    sample=f1_score(truth,prediction,average='samples',zero_division=0);macro=f1_score(truth,prediction,average='macro',zero_division=0)
    matched=pd.read_csv(out/'validation_predictions/L2_ratio0_C1000.csv')[[f'predicted_{r}' for r in reasons]].to_numpy(dtype=int)
    np.testing.assert_array_equal(prediction,matched)
    result={'purpose':'Native same-solver L2 tolerance diagnostic; not an additional penalty candidate','solver':'lbfgs','C':1000,'tol':1e-9,'max_iter':20000,
            'sample_f1':float(sample),'macro_f1':float(macro),'same_labels_as_full_batch_L2_C1000':True,'fit_audit':audit}
    (destination/'lbfgs_tight_C1000_metrics.json').write_text(json.dumps(result,indent=2)+'\n')
    joblib.dump(model,destination/'lbfgs_tight_C1000.joblib',compress=3)
    frame=valid[['ID']].copy()
    for j,reason in enumerate(reasons):frame[f'probability_{reason}']=probabilities[:,j];frame[f'predicted_{reason}']=prediction[:,j]
    frame.to_csv(destination/'lbfgs_tight_C1000_predictions.csv',index=False)
    comparison=pd.read_csv(out/'comparison.csv')
    best_grid=comparison.query("stage=='penalty_grid'").sort_values(['sample_f1','l1_fraction','C'],ascending=[False,True,True],kind='stable').groupby('penalty_type',sort=False).head(1)
    best_grid.to_csv(out/'best_converged_by_penalty.csv',index=False)
    settings=json.loads((out/'settings.json').read_text())
    settings['native_LBFGS_tolerance_diagnostic']='numerical_controls/lbfgs_tight_C1000_metrics.json'
    (out/'settings.json').write_text(json.dumps(settings,indent=2)+'\n')
    note=(f"The native scikit-learn LBFGS solver at the same C=1000, but tol=1e-9, gives sample F1 {sample:.6f} and macro F1 {macro:.6f}, matching the fully optimized L2 control's labels. The original tol=1e-4 model gives 0.910256/0.811421 and stops after 13–15 iterations; the tighter fits take 54–116. Thus stopping tolerance materially affects this model, and the incumbent is a particular fitted recipe rather than the tightly optimized L2 solution. Earlier stopping may be acting as implicit regularization; that interpretation is an inference. The current model and CSV remain unchanged. Best converged-by-penalty results are saved in best_converged_by_penalty.csv; compare those as well as the incumbent when assessing penalty effects.")
    summary=out/'SUMMARY.md';summary.write_text(summary.read_text()+'\n## Native solver tolerance diagnostic\n\n'+note+'\n')
    print(note)
    print(best_grid[['penalty_type','C','l1_fraction','sample_f1','macro_f1']].to_string(index=False))
    return result

native_l2_tolerance_result=verify_native_l2_tolerance()


The native scikit-learn LBFGS solver at the same C=1000, but tol=1e-9, gives sample F1 0.854701 and macro F1 0.703577, matching the fully optimized L2 control's labels. The original tol=1e-4 model gives 0.910256/0.811421 and stops after 13–15 iterations; the tighter fits take 54–116. Thus stopping tolerance materially affects this model, and the incumbent is a particular fitted recipe rather than the tightly optimized L2 solution. Earlier stopping may be acting as implicit regularization; that interpretation is an inference. The current model and CSV remain unchanged. Best converged-by-penalty results are saved in best_converged_by_penalty.csv; compare those as well as the incumbent when assessing penalty effects.
penalty_type      C  l1_fraction  sample_f1  macro_f1
          L2   10.0         0.00   0.867521  0.718678
  ElasticNet 2000.0         0.25   0.858974  0.708182
          L1    1.0         1.00   0.831197  0.634762
